In [ ]:

# CREDIT SCORING MODEL
# COMPLETE REFINED CODE


# IMPORT LIBRARIES

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE


# LOAD DATASET


df = pd.read_csv('/content/german_credit_data.csv')



# DISPLAY DATASET


print("FIRST 5 ROWS")
print(df.head())

print("\nDATASET SHAPE")
print(df.shape)

print("\nCOLUMNS")
print(df.columns)



# REMOVE UNNECESSARY COLUMN

if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)



# TARGET COLUMN ENCODING


df['Risk'] = df['Risk'].map({
    'good': 1,
    'bad': 0
})



# HANDLE MISSING VALUES


numerical_cols = df.select_dtypes(include=np.number).columns

categorical_cols = df.select_dtypes(include='object').columns

for col in numerical_cols:

    df[col] = df[col].fillna(
        df[col].median()
    )

for col in categorical_cols:

    df[col] = df[col].fillna(
        df[col].mode()[0]
    )


# FEATURE ENGINEERING


df['Credit_per_Duration'] = (
    df['Credit amount'] / df['Duration']
)

df['Is_Young'] = np.where(
    df['Age'] < 25,
    1,
    0
)

df['High_Credit'] = np.where(
    df['Credit amount'] > df['Credit amount'].median(),
    1,
    0
)



# REMOVE OUTLIERS


Q1 = df['Credit amount'].quantile(0.25)

Q3 = df['Credit amount'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR

upper = Q3 + 1.5 * IQR

df = df[
    (df['Credit amount'] >= lower)
    &
    (df['Credit amount'] <= upper)
]



# ONE HOT ENCODING


df = pd.get_dummies(
    df,
    drop_first=True
)


# FEATURES AND TARGET


X = df.drop('Risk', axis=1)

y = df['Risk']



# CLASS DISTRIBUTION


print("\nCLASS DISTRIBUTION")

print(y.value_counts())

sns.countplot(x=y)

plt.title("Class Distribution")

plt.show()


# TRAIN TEST SPLIT


X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

print("\nTrain Shape:", X_train.shape)

print("Test Shape:", X_test.shape)



# HANDLE CLASS IMBALANCE

smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(
    X_train,
    y_train
)

print("\nBalanced Class Distribution")

print(pd.Series(y_train).value_counts())



# FEATURE SCALING


scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)



# MODELS


models = {

    "Logistic Regression": LogisticRegression(

        max_iter=5000,

        solver='liblinear',

        C=0.2,

        class_weight='balanced',

        random_state=42
    ),

    "Random Forest": RandomForestClassifier(

        n_estimators=80,

        max_depth=4,

        min_samples_split=40,

        min_samples_leaf=20,

        max_features='sqrt',

        random_state=42
    ),

    "XGBoost": XGBClassifier(

        n_estimators=25,

        learning_rate=0.01,

        max_depth=2,

        subsample=0.6,

        colsample_bytree=0.6,

        reg_alpha=10,

        reg_lambda=15,

        gamma=5,

        random_state=42,

        eval_metric='logloss'
    )
}



# TRAINING AND EVALUATION

results = []

for name, model in models.items():

    print("\n================================================")
    print(f"MODEL: {name}")
    print("================================================")

    # TRAIN MODEL

    model.fit(X_train, y_train)



    # CROSS VALIDATION


    cv_scores = cross_val_score(

        model,

        X,

        y,

        cv=5,

        scoring='accuracy'
    )

    print(f"\nCross Validation Accuracy: {cv_scores.mean():.4f}")



    # OVERFITTING CHECK


    train_accuracy = model.score(
        X_train,
        y_train
    )

    test_accuracy = model.score(
        X_test,
        y_test
    )

    gap = train_accuracy - test_accuracy

    print(f"\nTraining Accuracy : {train_accuracy:.4f}")

    print(f"Testing Accuracy  : {test_accuracy:.4f}")

    print(f"Accuracy Gap      : {gap:.4f}")

    if gap < 0.05:

        print("Generalization: Excellent ✅")

    elif gap < 0.10:

        print("Generalization: Mild Overfitting ⚠️")

    else:

        print("Generalization: Strong Overfitting ❌")



    # PREDICTIONS


    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:, 1]


    # METRICS


    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred
    )

    recall = recall_score(
        y_test,
        y_pred
    )

    f1 = f1_score(
        y_test,
        y_pred
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    print("\nPerformance Metrics")

    print(f"Accuracy  : {accuracy:.4f}")

    print(f"Precision : {precision:.4f}")

    print(f"Recall    : {recall:.4f}")

    print(f"F1 Score  : {f1:.4f}")

    print(f"ROC-AUC   : {roc_auc:.4f}")



    # CLASSIFICATION REPORT

    print("\nClassification Report")

    print(

        classification_report(
            y_test,
            y_pred
        )
    )



    # CONFUSION MATRIX


    cm = confusion_matrix(
        y_test,
        y_pred
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm
    )

    disp.plot()

    plt.title(f"{name} Confusion Matrix")

    plt.show()



    # ROC CURVE


    RocCurveDisplay.from_estimator(

        model,

        X_test,

        y_test
    )

    plt.title(f"{name} ROC Curve")

    plt.show()



    # STORE RESULTS


    results.append({

        "Model": name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1,

        "ROC-AUC": roc_auc,

        "Train Accuracy": train_accuracy,

        "Test Accuracy": test_accuracy,

        "Gap": gap
    })



# FINAL RESULTS TABLE


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='Accuracy',
    ascending=False
)

print("\nFINAL RESULTS")

print(results_df)



# ACCURACY COMPARISON GRAPH


plt.figure(figsize=(10,5))

sns.barplot(

    x='Model',

    y='Accuracy',

    data=results_df
)

plt.title("Model Accuracy Comparison")

plt.ylim(0,1)

plt.show()